# Group 1 — Project 4 (push-ups: force plates + ultrasound)

Push-ups with a hand on **each force plate** and **ultrasound on the right forearm** (flexors, over radius & ulna) — tracked automatically with **DUSTrack**. Protocol: *discard the first two* (false start), then **5 normal** push-ups and **5 with torque** — for the torque set, watch **Mz** (the twisting moment) on both plates.

**Clocks** — force plates are on the **Motive clock**; the Delsys recorded only the **sync gate** (analog `Ch13`) so you can line the Delsys-timed events to Motive. The ultrasound has its own frame clock.

In [ ]:
!pip install -q delsys pysampled huggingface_hub

### 1 · Get your data

In [ ]:
from huggingface_hub import hf_hub_download
REPO = "praneethnamburimit/immersionlab-pe-mis-groups"
def get(p): return hf_hub_download(REPO, p, repo_type="dataset")
fp_L = get("group_1_4/bertec/group1_4_002_forceplate_1.csv")   # left arm
fp_R = get("group_1_4/bertec/group1_4_002_forceplate_2.csv")   # right arm
delsys_csv = get("group_1_4/delsys/Trial_2.csv")               # sync only
track_json = get("group_1_4/ultrasound_dustrack_tracking.json")
print("downloaded")

### 2 · Force plates — one per arm
`MotiveLog` (pasted; not on pip). `Fz` = push force, `Mz` = twisting moment.

In [ ]:
import csv as _csv, numpy as np, pandas as pd, pysampled
class MotiveLog:
    """Load a Motive force-plate 'Device export' CSV. MocapTime is the OptiTrack clock."""
    def __init__(self, fname):
        hdr = {}
        with open(fname, newline='') as f:
            for rc, row in enumerate(_csv.reader(f)):
                if not row: continue
                if row[0] == 'MocapFrame': break
                v = row[1] if len(row) > 1 else None
                try: hdr[row[0]] = float(v)
                except (TypeError, ValueError): hdr[row[0]] = v
        self.data = pd.read_csv(fname, skiprows=rc).rename(columns=lambda x: x.strip())
        self.sr = hdr['Mocap Rate'] * hdr.get('Mocap Rate Multiple', 1)
    def __getattr__(self, k):
        if k in ('Fx','Fy','Fz','Mx','My','Mz','Cx','Cy','Cz','Tz'):
            return pysampled.Data(self.data[k].values, sr=self.sr)
        raise AttributeError(k)

In [ ]:
import matplotlib.pyplot as plt
L = MotiveLog(fp_L); R = MotiveLog(fp_R)
t = np.arange(len(L.data))/L.sr                      # Motive clock
fig, ax = plt.subplots(2,1, figsize=(13,5), sharex=True)
ax[0].plot(t, np.abs(L.Fz()), label="left"); ax[0].plot(t, np.abs(R.Fz()), label="right")
ax[0].set_ylabel("|Fz| (N)"); ax[0].legend(); ax[0].set_title("push force per arm — count your 5 + 5 push-ups")
ax[1].plot(t, L.Mz(), label="left"); ax[1].plot(t, R.Mz(), label="right")
ax[1].set_ylabel("Mz (N·m)"); ax[1].set_xlabel("Motive time (s)"); ax[1].legend()
ax[1].set_title("twisting moment — the 'with torque' set should stand out here"); plt.tight_layout(); plt.show()

### 3 · Ultrasound — the DUSTrack tissue tracking
Two points were tracked on the forearm across every ultrasound frame; the **distance between them** is a simple tissue-deformation signal.

In [ ]:
import json
trk = json.load(open(track_json))
def track(pid):
    fr = sorted(trk[pid], key=int); v = [trk[pid][f] for f in fr]
    return np.array([x if isinstance(x, (list, tuple)) else [x['x'], x['y']] for x in v], float)
p0, p1 = track("0"), track("1")
tissue = np.linalg.norm(p0 - p1, axis=1)              # point-to-point distance, per ultrasound frame
plt.figure(figsize=(13,3)); plt.plot(tissue); plt.xlabel("ultrasound frame"); plt.ylabel("tissue distance (px)")
plt.title("forearm tissue deformation across the push-ups (ultrasound frames)"); plt.show()

### 4 · Synchronize (Delsys gate)
The Delsys analog channel marks when Motive started, so Delsys-clock events map to the Motive clock via `t_motive = t_delsys - offset`. (The ultrasound frame clock aligns through the same Delsys sync — a good stretch goal.)

In [ ]:
import numpy as np
def motive_gate_offset(lf):
    """Delsys analog sensor 13, channel 0 = Motive's record gate. Its rising edge is the
    Delsys-clock time when Motive (force plates + markers) began at t=0."""
    a = lf.find(sensor_number=13, as_="sensor")[0].analog
    g = np.asarray(a())[:, 0]; t = np.asarray(a.t)
    return float(t[np.argmax(g > 0.5*(g.min()+g.max()))])

In [ ]:
import delsys
lf = delsys.Log(delsys_csv); offset = motive_gate_offset(lf)
print(f"Motive started at Delsys t = {offset:.2f} s")

### 5 · Your turn
- Split into your **5 normal** and **5 torque** push-ups and compare **Mz** (twist) and the **left/right |Fz| balance** between the two sets.
- Line the **tissue-deformation** signal up with the force: does the forearm tissue deform most at the bottom of the push-up (peak force)?
- The raw ultrasound video + the `.tvd` are in your Dropbox folder if you want to re-track or eyeball the tissue.